# SDXL Convert Phase 2-5 (TPU v5e-8, 377GB RAM)

Consumes Phase 1 kernel's output (`data_sdxl.pkl` + `images_sdxl/`) from `/kaggle/input/sdxl-convert-phase1-gpu/phase1_out/`.

Runs:
- Phase 2: `gen_quant_data_sdxl.py`
- Phase 3: `export_onnx_sdxl.py`
- Phase 4+5: `convert_all_sdxl.sh` (qairt converter + quantizer)

TPU sits idle — we're using this runtime for its 330GB system RAM + 96 CPU cores.

In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
MIN_SOC            = '__MIN_SOC__'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC} min_soc={MIN_SOC}')

In [ ]:
!free -h
!nproc
!df -h /kaggle/working /tmp
!cat /etc/os-release | head -5
!ldd --version | head -1
!grep -m1 'model name' /proc/cpuinfo
!grep -m1 'cpu MHz' /proc/cpuinfo
!lscpu | grep -E 'Model name|Socket|Thread|Core|Vendor|Flags' | head -10

In [ ]:
# Locate Phase 1 output
import glob, os
candidates = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert candidates, 'Phase 1 data_sdxl.pkl not found in /kaggle/input/ — did Phase 1 kernel complete?'
PHASE1_DIR = os.path.dirname(candidates[0])
print(f'Phase 1 dir: {PHASE1_DIR}')
!ls -lh $PHASE1_DIR

In [ ]:
import os
os.environ['MIN_SOC'] = MIN_SOC
os.environ['MODEL_NAME'] = MODEL_NAME

In [ ]:
# Install tools + convertsdxl + shell helper
import os, subprocess

def _run(cmd, log=None, check=True):
    """bash with errexit + pipefail; pipe to tee if log. Raises on non-zero."""
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        wrapped = f'set -eo pipefail; {cmd} 2>&1 | tee {log}'
    else:
        wrapped = f'set -eo pipefail; {cmd}'
    rc = subprocess.call(['bash', '-c', wrapped])
    if check and rc != 0:
        raise RuntimeError(f'shell failed (rc={rc}): {cmd[:200]}  log={log}')
    return rc

# Ensure apt cache is fresh before any installs
_run('apt-get update -y > /dev/null')
_run('apt-get install -y unzip zip curl time > /dev/null')
# libc++/libc++abi/libunwind needed by QNN SDK libPyIrGraph.so / libQnnHtp.so — Kaggle Debian 13 trixie doesn't ship them
# Specific LLVM runtime packages — generic libc++-dev/libunwind-dev do NOT provide libunwind.so.1 (GNU libunwind ships .so.8)
_run('apt-get install -y libc++1-19 libc++abi1-19 libunwind-19 > /dev/null')
_run('ldconfig')  # refresh ld cache so newly installed libs are found by ldd / dlopen
_run('curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1')
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

_run('rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip')
_run("curl -sL --fail --retry 5 --retry-delay 5 -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'")
_run('mkdir -p /tmp/convertsdxl_unzip')
_run('unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip')

root = subprocess.check_output(
    "find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1",
    shell=True, text=True).strip()
assert root, 'export_sdxl.sh not found in unzipped convertsdxl'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# QNN SDK
import os, subprocess
_run('mkdir -p /tmp/qnn_sdk')
_run(
    "curl -sL --fail --retry 5 --retry-delay 5 -A 'Mozilla/5.0' "
    "-o /tmp/qnn_sdk/qnn.zip "
    "'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip'"
)
_run('cd /tmp/qnn_sdk && unzip -q qnn.zip')
bin_file = subprocess.check_output(
    'find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit',
    shell=True, text=True).strip()
assert bin_file, 'envsetup.sh not found in QNN SDK extract'
QNN_SDK_BIN = os.path.dirname(bin_file)
QNN_SDK_ROOT_DIR = os.path.dirname(QNN_SDK_BIN)
os.environ['QNN_SDK_BIN'] = QNN_SDK_BIN
os.environ['QNN_SDK_ROOT_DIR'] = QNN_SDK_ROOT_DIR
print(f'QNN_SDK_ROOT: {QNN_SDK_ROOT_DIR}')

In [ ]:
# Resolve model.safetensors from Phase 1 kernel_sources mount (read-only, no re-download)
import os, glob
candidates = glob.glob('/kaggle/input/**/model.safetensors', recursive=True)
assert candidates, 'model.safetensors not found in /kaggle/input/ — Phase 1 kernel_sources not mounted?'
os.environ['MODEL_PATH'] = candidates[0]
size = os.path.getsize(os.environ['MODEL_PATH'])
assert size > int(1e9), f'model.safetensors truncated: {size} bytes'
print(f'MODEL_PATH -> {os.environ["MODEL_PATH"]} ({size/1e9:.2f} GB)')

In [ ]:
# Python env
import os
os.chdir(os.environ['NPUCONVERT_DIR'])
_run('uv venv -p 3.10.17 --clear')
_run('. .venv/bin/activate && uv sync')

In [ ]:
# Copy Phase 1 output into convert dir
import os, shutil, glob
pkl_list = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert pkl_list, 'data_sdxl.pkl not found in /kaggle/input/ — Phase 1 mount missing'
pkl = pkl_list[0]
p1_dir = os.path.dirname(pkl)
shutil.copy(pkl, f'{os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
src_img = f'{p1_dir}/images_sdxl'
assert os.path.isdir(src_img), f'{src_img} missing — Phase 1 did not produce calibration images'
dst_img = f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl'
if os.path.exists(dst_img):
    shutil.rmtree(dst_img)
shutil.copytree(src_img, dst_img)
assert len(os.listdir(dst_img)) > 0, f'{dst_img} empty after copy'
_run(f'ls -lh {os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
_run(f'ls {dst_img} | head -5')

In [ ]:
# TPU burner: torch_xla Llama-3.2-1B LoRA SFT on GSM8K — keeps TPU continuously busy
# so Kaggle's 2h-idle auto-cancel doesn't trigger during long CPU-only convert Phase 4-5.
# Architecture:
#   - ONE subprocess.Popen call, runs entire notebook lifetime (no periodic respawn)
#   - torch_xla[tpu]==2.6.0 on v5e-8 (pytorch/xla + cloud.google docs verified — 2.6 is earliest with cp312 wheels)
#   - Signal per round = /kaggle/working/burner/rounds/round_NNNN.txt  (HW5 cell 32 semantics)
#   - Train data pre-tokenized once (fixes old tf.random.normal per-step CPU churn)
#   - bf16 on TPU HBM, no bitsandbytes (CUDA-only) — Llama-1B ~2.5GB fits in 16GB/chip
import subprocess, os, time, textwrap

os.makedirs('/kaggle/working/logs', exist_ok=True)
os.makedirs('/kaggle/working/burner/rounds', exist_ok=True)
os.makedirs('/kaggle/working/burner/data', exist_ok=True)

# -- mem watcher (unchanged from baseline) --
watcher_sh = """#!/bin/bash
echo "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"
while true; do
  epoch=$(date +%s); dt=$(date -Iseconds)
  mem=$(awk '/MemAvailable:/{print int($2/1024)}' /proc/meminfo)
  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)
  rss=$(echo "$line" | awk '{print int($1/1024)}')
  proc=$(echo "$line" | awk '{print $2}')
  echo "$epoch,$dt,$mem,$rss,$proc"
  sleep 10
done
"""
open('/tmp/mem_watch.sh','w').write(watcher_sh)
os.chmod('/tmp/mem_watch.sh', 0o755)
p = subprocess.Popen(['/tmp/mem_watch.sh'],
    stdout=open('/kaggle/working/logs/mem-trace.csv','w'), stderr=subprocess.DEVNULL)
open('/tmp/watcher.pid','w').write(str(p.pid))
print(f'[*] {time.strftime("%H:%M:%S")} mem-watcher pid={p.pid}')

# -- Build ISOLATED uv venv 3.10 for burner (torch_xla[tpu]==2.6.0 only ships cp39/10/11 wheels on libtpu-releases; Kaggle TPU system python is 3.12)
BURNER_VENV = '/tmp/burner_venv'  # /tmp has hundreds of GB; /kaggle/working is 20GB cap (phase3_backup eats 13GB)
BURNER_PY_BIN = f'{BURNER_VENV}/bin/python'
print(f'[*] {time.strftime("%H:%M:%S")} burner venv (py3.10) create start')
_run(f'uv venv -p 3.10 "{BURNER_VENV}" --clear 2>&1 | tail -3')
_run(f'uv pip install --python "{BURNER_PY_BIN}" --quiet '
     "torch==2.6.0 \"torch_xla[tpu]==2.6.0\" "
     '-f https://storage.googleapis.com/libtpu-releases/index.html '
     '2>&1 | tail -5')
_run(f'uv pip install --python "{BURNER_PY_BIN}" --quiet '
     'transformers==4.46.3 peft==0.14.0 datasets==3.2.0 accelerate==1.2.1 huggingface_hub gdown '
     '2>&1 | tail -3')
print(f'[*] {time.strftime("%H:%M:%S")} burner venv install end')

# -- Smoke test: torch_xla sees TPU --
print(f'[*] {time.strftime("%H:%M:%S")} torch_xla smoke test start')
probe = subprocess.run(
    [BURNER_PY_BIN, '-c',
     'import os; os.environ.setdefault("PJRT_DEVICE","TPU"); '
     'import torch, torch_xla.core.xla_model as xm; '
     'd=xm.xla_device(); t=torch.ones(1024,1024,device=d); '
     'print("DEV="+str(d)); '
     'print("N_TPU="+str(len(xm.get_xla_supported_devices()))); '
     'print("SUM={:.2e}".format(float((t@t).sum().cpu())))'
    ],
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PJRT_DEVICE':'TPU'})
print(f'[*] {time.strftime("%H:%M:%S")} torch_xla smoke test end rc={probe.returncode}')
print(f'stdout: {probe.stdout.strip()[-400:]}')
print(f'stderr[-400:]: {probe.stderr.strip()[-400:]}')
if probe.returncode != 0 or 'N_TPU=' not in probe.stdout:
    raise RuntimeError(f'torch_xla probe failed; TPU unreachable. stderr:\n{probe.stderr[-1500:]}')
# Parse + log TPU count as visible milestone
_ntpu = _dev = '?'
for _ln in probe.stdout.splitlines():
    if _ln.startswith('N_TPU='): _ntpu = _ln.split('=',1)[1].strip()
    elif _ln.startswith('DEV='):  _dev  = _ln.split('=',1)[1].strip()
print(f'[*] {time.strftime("%H:%M:%S")} TPU available: device={_dev} chips={_ntpu}')

# -- HF_TOKEN from GHA secret (placeholder patched by workflow step) --
HF_TOKEN = '__HF_TOKEN__'
if HF_TOKEN and not HF_TOKEN.startswith('__'):
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print(f'[*] {time.strftime("%H:%M:%S")} HF_TOKEN injected (len={len(HF_TOKEN)})')
else:
    print(f'[!] {time.strftime("%H:%M:%S")} HF_TOKEN not provided — gated Llama-3.2-1B download will 401')

# -- Download HW5 datasets (5 files from HW5 cell 6) --
DATA_DIR = '/kaggle/working/burner/data'
GDOWN = [
    ('1KmElxmqj-xXIJi7vc9V7TmbVWDICynEb','gsm8k_train.jsonl'),
    ('12AShd7x6IWZurCCz_16jXxInt5cV8aRM','gsm8k_train_self-instruct.jsonl'),
    ('1DMd8zJs1lfRjc0TISsdwU4JJuL-CrCdc','gsm8k_test_public.jsonl'),
    ('1eAk06RDJVFosQ0FEQ3KdOssJW-Q9zfVr','gsm8k_test_private.jsonl'),
    ('1UYJZxvQQUyGh--qI2d1kCHOSRZhtGtdk','ailuminate_test.csv'),
]
print(f'[*] {time.strftime("%H:%M:%S")} HW5 dataset download start')
for gid, fname in GDOWN:
    fp = f'{DATA_DIR}/{fname}'
    if os.path.exists(fp) and os.path.getsize(fp) > 100: continue
    _run(f'gdown --quiet "https://drive.google.com/uc?id={gid}" -O "{fp}"')
print(f'[*] {time.strftime("%H:%M:%S")} HW5 dataset download end')
_run(f'ls -lh {DATA_DIR}')

# -- Write burner.py (self-contained Llama LoRA SFT loop, HW5-style round signal txt) --
BURNER_PY = r'''"""Llama-3.2-1B LoRA SFT on GSM8K via torch_xla. Infinite loop, one signal txt per round."""
import os, sys, json, time, random, csv, traceback, signal as _signal
os.environ.setdefault("PJRT_DEVICE","TPU")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATA = "/kaggle/working/burner/data"
ROUNDS = "/kaggle/working/burner/rounds"
os.makedirs(ROUNDS, exist_ok=True)

def log(msg):
    print(f"[burner {time.strftime('%H:%M:%S')}] {msg}", flush=True)

# SIGTERM handler — cell 15 finally-block sends SIGTERM before SIGKILL
_signal.signal(_signal.SIGTERM, lambda *_: (log("SIGTERM received — exit"), sys.exit(0)))

log("startup — importing torch + torch_xla")
import torch
import torch_xla.core.xla_model as xm
DEV = xm.xla_device()
log(f"torch={torch.__version__} xla_device={DEV} n_tpu={len(xm.get_xla_supported_devices())}")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL = "meta-llama/Llama-3.2-1B-Instruct"
log(f"loading tokenizer {MODEL}")
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

log(f"loading model {MODEL} bf16 (no bitsandbytes on TPU)")
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16
)
log(f"base loaded in {time.time()-t0:.1f}s params={base.num_parameters():,}")

lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["up_proj","down_proj","gate_proj","k_proj","q_proj","v_proj","o_proj"],
)
model = get_peft_model(base, lora_cfg).to(DEV)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"peft lora ready on {DEV}; trainable={trainable:,}")

def load_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f]
gsm_train = load_jsonl(f"{DATA}/gsm8k_train.jsonl")
gsm_pub = load_jsonl(f"{DATA}/gsm8k_test_public.jsonl")
gsm_prv = load_jsonl(f"{DATA}/gsm8k_test_private.jsonl")
with open(f"{DATA}/ailuminate_test.csv") as f:
    aim = [r["prompt_text"] for r in csv.DictReader(f)]
log(f"data: train={len(gsm_train)} pub={len(gsm_pub)} prv={len(gsm_prv)} aim={len(aim)}")

N_SHOT = 2        # scaled up from HW5 default 1
MAX_LEN = 512
BATCH = 2
GRAD_ACCUM = 4
LR = 1e-4         # scaled down from HW5 default 2e-4 per TODO in cell 21

def nshot_chats(data, n, question, answer, mode):
    chats = []
    for qna in random.sample(data, n):
        chats.append({"role":"user","content":f"Q: {qna['question']}"})
        chats.append({"role":"assistant","content":f"A: {qna['answer']}"})
    instr = "Q: " + question + " Let's think step by step. At the end, you MUST write the answer as an integer after '####'."
    chats.append({"role":"user","content":instr})
    if mode == "train":
        chats.append({"role":"assistant","content":f"A: {answer}"})
    return chats

# Pre-tokenize train set ONCE (no per-step CPU churn)
random.seed(42)
TRAIN_CAP = 500
log(f"formatting + tokenizing {TRAIN_CAP} train samples (max_len={MAX_LEN})")
texts = []
for qna in gsm_train[:TRAIN_CAP]:
    chats = nshot_chats(gsm_train, N_SHOT, qna["question"], qna["answer"], "train")
    t = tok.apply_chat_template(chats, tokenize=False)
    i = t.find("<|eot_id|>")
    if i >= 0: t = t[i+len("<|eot_id|>"):]
    texts.append(t)
enc = tok(texts, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
IDS, ATT = enc["input_ids"], enc["attention_mask"]
LBL = IDS.clone(); LBL[ATT==0] = -100
log(f"tokenized: input_ids.shape={tuple(IDS.shape)}")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

def train_one_epoch(epoch_n):
    model.train()
    n = IDS.size(0)
    order = list(range(n))
    random.Random(epoch_n).shuffle(order)
    total, steps = 0.0, 0
    for i in range(0, n, BATCH):
        bi = order[i:i+BATCH]
        ii = IDS[bi].to(DEV); am = ATT[bi].to(DEV); lb = LBL[bi].to(DEV)
        out = model(input_ids=ii, attention_mask=am, labels=lb)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        total += float(loss.detach().cpu()) * GRAD_ACCUM
        steps += 1
        if steps % GRAD_ACCUM == 0:
            xm.optimizer_step(opt, barrier=True)
            opt.zero_grad()
        if steps % 50 == 0:
            xm.mark_step()
            log(f"epoch={epoch_n} step={steps}/{(n+BATCH-1)//BATCH} avg_loss={total/steps:.4f}")
    xm.optimizer_step(opt, barrier=True); opt.zero_grad()
    return total / max(steps, 1)

@torch.no_grad()
def gen(prompt_chats, max_new=128):
    model.eval()
    txt = tok.apply_chat_template(prompt_chats, tokenize=False, add_generation_prompt=True)
    enc2 = tok(txt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    ids = enc2.input_ids.to(DEV)
    prompt_len = ids.size(1)
    eos = tok.eos_token_id
    for _ in range(max_new):
        logits = model(input_ids=ids).logits[:, -1, :]
        nxt = logits.argmax(-1, keepdim=True)
        ids = torch.cat([ids, nxt], dim=1)
        if int(nxt.cpu().item()) == eos:
            break
    xm.mark_step()
    out = ids[0, prompt_len:].cpu()
    return tok.decode(out, skip_special_tokens=True)

def extract_ans(resp):
    a = resp.split("####")[-1].strip()
    for c in [",","$","%","g"]: a = a.replace(c, "")
    return a

round_n = 0
while True:
    round_n += 1
    log(f"=== ROUND {round_n} start ===")
    try:
        loss = train_one_epoch(round_n)
        log(f"round={round_n} train_done loss={loss:.4f}")

        random.seed(round_n)
        gsm_preds = []; correct = 0
        PUB_CAP = 100
        for i, qna in enumerate(gsm_pub[:PUB_CAP]):
            msgs = nshot_chats(gsm_train, N_SHOT, qna["question"], None, "test")
            r = gen(msgs)
            pa = extract_ans(r); ta = extract_ans(qna["answer"])
            if pa == ta: correct += 1
            gsm_preds.append(pa)
            if (i+1) % 20 == 0:
                log(f"round={round_n} gsm_pub {i+1}/{PUB_CAP} acc={correct/(i+1):.3f}")
        log(f"round={round_n} gsm_pub_final_acc={correct/PUB_CAP:.3f}")

        PRV_CAP = 50
        for qna in gsm_prv[:PRV_CAP]:
            msgs = nshot_chats(gsm_train, N_SHOT, qna["question"], None, "test")
            gsm_preds.append(extract_ans(gen(msgs)))
        log(f"round={round_n} gsm_prv_done n={PRV_CAP}")

        aim_preds = []
        AIM_CAP = 20
        for q in aim[:AIM_CAP]:
            aim_preds.append(gen([{"role":"user","content":q}], max_new=200))
        log(f"round={round_n} aim_done n={AIM_CAP}")

        sigf = f"{ROUNDS}/round_{round_n:04d}.txt"
        with open(sigf, "w") as f:
            print(gsm_preds + aim_preds, file=f)
        log(f"round={round_n} signal={sigf}")

    except Exception as e:
        log(f"round={round_n} ERROR: {e!r}")
        traceback.print_exc()
        time.sleep(30)
'''
open('/kaggle/working/burner/burner.py','w').write(BURNER_PY)
print(f'[*] {time.strftime("%H:%M:%S")} burner.py written ({len(BURNER_PY.splitlines())} lines)')

# -- Launch burner subprocess ONCE (runs for entire notebook lifetime) --
burner_log = '/kaggle/working/logs/burner.log'
bp_env = {**os.environ, 'PJRT_DEVICE': 'TPU'}
bp = subprocess.Popen(
    [BURNER_PY_BIN, '/kaggle/working/burner/burner.py'],
    stdout=open(burner_log,'w'), stderr=subprocess.STDOUT,
    env=bp_env, cwd='/kaggle/working/burner',
)
open('/tmp/burner.pid','w').write(str(bp.pid))
print(f'[*] {time.strftime("%H:%M:%S")} burner pid={bp.pid} launched (ONE subprocess, full lifetime)')

# -- Wait for burner readiness (peft lora ready log line) --
print(f'[*] {time.strftime("%H:%M:%S")} waiting up to 10min for burner model load...')
t_wait = time.time()
ready = False
while time.time() - t_wait < 600:
    if bp.poll() is not None:
        tail = open(burner_log).read()[-2500:]
        raise RuntimeError(f'burner died early rc={bp.returncode}:\n{tail}')
    try:
        log_txt = open(burner_log).read()
    except FileNotFoundError:
        log_txt = ''
    if 'peft lora ready on' in log_txt:
        ready = True
        break
    time.sleep(15)
if not ready:
    raise RuntimeError(f'burner not ready in 10min. Log tail:\n' + open(burner_log).read()[-2500:])
print(f'[*] {time.strftime("%H:%M:%S")} burner ready after {int(time.time()-t_wait)}s')
_run(f'tail -20 {burner_log}', check=False)


In [ ]:
# Phase 2 — full log in phase2.log; stdout only key milestones
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
print(f'[*] {time.strftime("%H:%M:%S")} Phase 2 start')
_run(
    '. .venv/bin/activate && /usr/bin/time -v python gen_quant_data_sdxl.py 2>&1 '
    '| tee /kaggle/working/logs/phase2.log '
    '| awk \'/^\\[\\*\\]|^===|ERROR|Traceback|Saved|Elapsed \\(wall clock\\)|Maximum resident set size/\'',
    log=None
)
print(f'[*] {time.strftime("%H:%M:%S")} Phase 2 end elapsed={int(time.time()-t0)}s')
_run('free -h')


In [ ]:
# Phase 3 — TracerWarning flood filtered; full log in phase3.log
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
print(f'[*] {time.strftime("%H:%M:%S")} Phase 3 start')
_run(
    '. .venv/bin/activate && /usr/bin/time -v python export_onnx_sdxl.py --model_path "$MODEL_PATH" 2>&1 '
    '| tee /kaggle/working/logs/phase3.log '
    '| awk \'/^\\[\\*\\]|^===|ERROR|Traceback|Saved|Elapsed \\(wall clock\\)|Maximum resident set size/\'',
    log=None
)
print(f'[*] {time.strftime("%H:%M:%S")} Phase 3 end elapsed={int(time.time()-t0)}s')
_run('free -h')
_run(r'find . -name "*.onnx" -exec ls -lh {} \;')
_run(r'find . -name "weights.pb" -exec ls -lh {} \;')


In [ ]:
# Backup Phase 3 ONNX output to /kaggle/working/phase3_backup
import os, shutil
bkp = '/kaggle/working/phase3_backup'
if os.path.exists(bkp): shutil.rmtree(bkp)
os.makedirs(bkp)
expected = ['clip_sdxl','clip2_sdxl','unet_sdxl','vae_decoder_sdxl','vae_encoder_sdxl']
missing = [d for d in expected if not os.path.exists(f'{os.environ["NPUCONVERT_DIR"]}/{d}')]
assert not missing, f'Phase 3 output dirs missing (partial export?): {missing}'
for d_ in expected:
    shutil.copytree(f'{os.environ["NPUCONVERT_DIR"]}/{d_}', f'{bkp}/{d_}')
_run(f'ls {bkp}')
_run(f'du -sh {bkp}/*')
_run(rf'find {os.environ["NPUCONVERT_DIR"]} -maxdepth 2 -name "input_list_*.txt" -exec cp {{}} {bkp}/ \;')
_run(f'ls {bkp}/*.txt', check=False)  # .txt copy is cosmetic

In [ ]:
# Patch convert scripts: replace hardcoded QNN_SDK_ROOT, make all config paths ABSOLUTE to NPUCONVERT_DIR.
# Reason: qnn-context-binary-generator's --config_file ./x.json is cwd-relative; if cwd drifts between
# qairt-quantizer (22min) and qnn-context-binary-generator we fail with "Cannot open file".
# Absolute paths eliminate the entire class of problems.
import os, re, glob, json as _json

NPU = os.environ['NPUCONVERT_DIR']
QNN = os.environ['QNN_SDK_ROOT_DIR']
SOC = os.environ['MIN_SOC']  # e.g. '8gen3' or '8gen4' (upstream default moved 8gen4 -> 8gen3 on 2026-04-20)

# 1. convert_all_sdxl.sh: replace hardcoded QNN_SDK_ROOT + prepend cd to NPUCONVERT_DIR
main = f'{NPU}/scripts/convert_all_sdxl.sh'
orig = open(main).read()
pat = re.compile(r'QNN_SDK_ROOT=/data/qairt/[0-9.]+')
assert pat.search(orig), f'QNN_SDK_ROOT pattern not found in {main}'
patched = pat.sub(f'QNN_SDK_ROOT={QNN}', orig)
patched = patched.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
assert patched != orig
open(main, 'w').write(patched)

# 2. Each sub-script: prepend defensive cd
for sub in glob.glob(f'{NPU}/scripts/convert_*.sh'):
    if sub.endswith('convert_all_sdxl.sh'):
        continue
    s = open(sub).read()
    s2 = s.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
    open(sub, 'w').write(s2)
    print(f'patched {os.path.basename(sub)}: prepended cd')

# 3. Rewrite htp_backend_<SOC>.json so config_file_path is absolute
hbp = f'{NPU}/htp_backend_{SOC}.json'
assert os.path.exists(hbp), f'{hbp} missing — upstream convertsdxl.zip may not ship this SOC config'
d = _json.load(open(hbp))
d['backend_extensions']['config_file_path'] = f'{NPU}/htp_config_{SOC}.json'
_json.dump(d, open(hbp, 'w'), indent=2)
print(f'htp_backend_{SOC}.json now:', open(hbp).read())

# 4. Sanity grep
_run(f'grep -nE "QNN_SDK_ROOT|^cd |source envsetup" {main}')

In [ ]:
# Phase 4-5 — QNN convert + quantize, with burner kill + package in finally block.
# Per user Q4: when convert ends (success OR failure), stop burner and output bin.
import time, os, subprocess, signal, shutil
t0 = time.time()
SOC = os.environ['MIN_SOC']
NPU = os.environ['NPUCONVERT_DIR']
os.chdir(NPU)  # ensure relative paths (.venv/bin/python, output/...) resolve

def _kill_pidfile(pidfile, name):
    """Two-stage kill: SIGTERM then SIGKILL after 5s. Safe in finally (no raise)."""
    try:
        pid = int(open(pidfile).read().strip())
    except (FileNotFoundError, ValueError):
        return
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'[*] {time.strftime("%H:%M:%S")} SIGTERM pid={pid} ({name})')
    except ProcessLookupError:
        return
    time.sleep(5)
    try:
        os.kill(pid, signal.SIGKILL)
        print(f'[*] {time.strftime("%H:%M:%S")} SIGKILL pid={pid} ({name})')
    except ProcessLookupError:
        pass

convert_ok = False
try:
    # LD_LIBRARY_PATH: qairt-converter's libPyIrGraph.so needs libpython3.10.so.1.0 from uv venv
    _pylib = subprocess.check_output(
        ['.venv/bin/python', '-c', 'import sys,os; print(os.path.join(sys.base_prefix, "lib"))'],
        text=True).strip()
    assert os.path.exists(os.path.join(_pylib, 'libpython3.10.so.1.0')), \
        f'libpython3.10.so.1.0 not in {_pylib}'
    os.environ['LD_LIBRARY_PATH'] = f"{_pylib}:{os.environ.get('LD_LIBRARY_PATH','')}"

    # Pre-flight: QNN .so deps all resolve
    _qnn = os.environ['QNN_SDK_ROOT_DIR']
    _qnn_lib = f'{_qnn}/lib/x86_64-linux-clang'
    _chk_env = os.environ.copy()
    _chk_env['LD_LIBRARY_PATH'] = f'{_qnn_lib}:{_chk_env["LD_LIBRARY_PATH"]}'
    _critical_libs = [
        f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrGraph.so',
        f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrQuantizer.so',
        f'{_qnn}/lib/x86_64-linux-clang/libQnnHtp.so',
        f'{_qnn}/lib/x86_64-linux-clang/libQnnModelDlc.so',
    ]
    for _lib in _critical_libs:
        assert os.path.exists(_lib), f'{_lib} not found in SDK extract'
        _ldd = subprocess.check_output(['ldd', _lib], env=_chk_env, text=True)
        _missing = [l.strip() for l in _ldd.splitlines() if 'not found' in l]
        if _missing:
            raise AssertionError(f'{_lib} unresolved deps:\n' + '\n'.join(_missing) + f'\n{_ldd}')

    # Pre-flight: required files exist
    _required = [
        f'{NPU}/htp_backend_{SOC}.json', f'{NPU}/htp_config_{SOC}.json',
        f'{NPU}/tokenizer.json', f'{NPU}/MNNConvert',
        f'{NPU}/scripts/convert_all_sdxl.sh', f'{NPU}/data_sdxl.pkl',
        f'{NPU}/vae_encoder_sdxl/model.onnx', f'{NPU}/vae_decoder_sdxl/model.onnx',
        f'{NPU}/unet_sdxl/model.onnx', f'{NPU}/clip_sdxl/model.onnx', f'{NPU}/clip2_sdxl/model.onnx',
        f'{NPU}/input_list_unet_sdxl.txt', f'{NPU}/input_list_vae_decoder_sdxl.txt',
        f'{NPU}/input_list_vae_encoder_sdxl.txt',
    ]
    _missing_files = [p for p in _required if not os.path.exists(p)]
    assert not _missing_files, 'Required files missing:\n' + '\n'.join(_missing_files)
    print(f'[*] {time.strftime("%H:%M:%S")} pre-flight OK ({len(_required)} files, {len(_critical_libs)} libs)')

    # Stdout whitelist — only key milestones + errors reach Kaggle UI; full output in phase45.log
    os.makedirs('/kaggle/working/logs', exist_ok=True)
    print(f'[*] {time.strftime("%H:%M:%S")} convert_all_sdxl.sh start')
    _run(
        f'cd "{NPU}" && . .venv/bin/activate && '
        f'(/usr/bin/time -v bash scripts/convert_all_sdxl.sh --min_soc "$MIN_SOC") 2>&1 '
        f'| tee /kaggle/working/logs/phase45.log '
        f'| awk \'/^\\[\\*\\]|^===|ERROR|Traceback|INFO_CONVERSION_SUCCESS|Quantization completed successfully|Quantized Model saved at|INFO_WRITE_SUCCESS|Converted Success!|Model larger than 2GB|Saved|Elapsed \\(wall clock\\)|Maximum resident set size/\'',
        log=None
    )
    convert_ok = True
    print(f'[*] {time.strftime("%H:%M:%S")} convert_all_sdxl.sh end elapsed={int(time.time()-t0)}s')

finally:
    # (1) Stop burner subprocess + mem watcher (per Q4 — convert end → kill → package)
    print(f'[*] {time.strftime("%H:%M:%S")} finally: stop burner + mem watcher (convert_ok={convert_ok})')
    _kill_pidfile('/tmp/burner.pid', 'burner')
    _kill_pidfile('/tmp/watcher.pid', 'mem-watcher')

    # (2) Package whatever output exists
    os.chdir(NPU)
    zip_path = f'/kaggle/working/{os.environ["MODEL_NAME"]}_qnn2.28_{SOC}.zip'
    out_dir = f'output/qnn_models_sdxl_{SOC}'
    if convert_ok and os.path.isdir(out_dir):
        bkp = '/kaggle/working/phase3_backup'
        if os.path.exists(bkp):
            shutil.rmtree(bkp)
            print(f'[*] {time.strftime("%H:%M:%S")} reclaimed phase3_backup (convert succeeded)')
        open(f'{out_dir}/SDXL', 'w').close()
        rc = subprocess.call(['zip', '-r', zip_path, out_dir], stdout=subprocess.DEVNULL)
        print(f'[*] {time.strftime("%H:%M:%S")} packaged -> {zip_path} (zip rc={rc})')
    else:
        print(f'[*] {time.strftime("%H:%M:%S")} convert failed — phase3_backup retained, no final zip')
    subprocess.call(['df', '-h', '/kaggle/working'])
    print(f'[*] {time.strftime("%H:%M:%S")} Phase 4-5 total elapsed={int(time.time()-t0)}s convert_ok={convert_ok}')


In [ ]:
# Summary: mem usage profile + burner round completions + output listing
import pandas as pd, os, glob, time
print(f'[*] {time.strftime("%H:%M:%S")} summary start')

if os.path.exists('/kaggle/working/logs/mem-trace.csv'):
    df = pd.read_csv('/kaggle/working/logs/mem-trace.csv')
    print(f'mem samples: {len(df)}')
    if len(df):
        print('5 lowest MemAvail (MB):')
        print(df.nsmallest(5,'MemAvail_MB')[['datetime','MemAvail_MB','TopRSS_MB','TopProc']].to_string())
        print('5 highest TopRSS (MB):')
        print(df.nlargest(5,'TopRSS_MB')[['datetime','TopRSS_MB','TopProc']].to_string())

rounds = sorted(glob.glob('/kaggle/working/burner/rounds/round_*.txt'))
print(f'burner rounds completed: {len(rounds)}')
if rounds:
    print(f'  first: {os.path.basename(rounds[0])}')
    print(f'  last:  {os.path.basename(rounds[-1])}')

_run('ls -lh /kaggle/working/', check=False)
_run('ls -lh /kaggle/working/logs/', check=False)

# tail burner log — 顯示 LoRA SFT 跑了多少輪、最後狀態
if os.path.exists('/kaggle/working/logs/burner.log'):
    print('--- burner.log tail (last 30 lines) ---')
    _run('tail -30 /kaggle/working/logs/burner.log', check=False)
